# Notebook 1: Factorization Machine (FM) from Scratch
### iCog Labs Intern Training: Predicting Risky Drug Combinations

**Task:** Build an FM from scratch that predicts whether the interaction between two drugs is **Severe** (`Label = 1`) or **Not severe** (`Label = 0`).

**Rules:** `pandas`, `numpy`, `matplotlib`; from `scikit-learn` only `train_test_split` and metrics.  
No PyTorch, TensorFlow, or any FM/recommender library.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, os, csv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)

SEED = 42
np.random.seed(SEED)
print('Libraries loaded. SEED =', SEED)

## 2. Load and explore the data

In [ ]:
# Upload db_drug_interactions_severity.csv using the Files panel (Colab)
# or place it in the same directory.
df = pd.read_csv('db_drug_interactions_severity.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

print('\nLabel distribution:')
counts = df['Label'].value_counts()
pcts   = df['Label'].value_counts(normalize=True) * 100
print(pd.DataFrame({'count': counts, '%': pcts.round(2)}))

## 3. Choose inputs and target

In [ ]:
# Inputs: Drug 1 and Drug 2 ONLY.
# Target: Label.  (Interaction Description and Severity are NEVER inputs.)
X_raw = df[['Drug 1', 'Drug 2']].copy()
y     = df['Label'].values.astype(np.float32)

print('X shape:', X_raw.shape)
print('y shape:', y.shape)
print('Severe (y=1):', y.sum(), '  Not severe (y=0):', (y==0).sum())

## 4. Train / validation / test split

In [ ]:
# ── Step 1: build canonical pair key so (A,B) and (B,A) stay together ──
df2 = df[['Drug 1', 'Drug 2', 'Label']].copy()
df2['_pair_key'] = df2.apply(
    lambda r: tuple(sorted([r['Drug 1'], r['Drug 2']])), axis=1)

# ── Step 2: one-row-per-pair for stratified splitting ───────────────────
pair_summary = (
    df2.groupby('_pair_key')['Label']
    .agg(lambda s: int(s.mode()[0]))
    .reset_index()
    .rename(columns={'Label': '_strat'})
)
keys  = pair_summary['_pair_key'].values
strat = pair_summary['_strat'].values

# ── Step 3: split pair keys ─────────────────────────────────────────────
train_val_keys, test_keys, tv_strat, _ = train_test_split(
    keys, strat, test_size=0.15, stratify=strat, random_state=SEED)

val_ratio_adj = 0.15 / (1.0 - 0.15)
train_keys, val_keys, _, _ = train_test_split(
    train_val_keys, tv_strat,
    test_size=val_ratio_adj, stratify=tv_strat, random_state=SEED)

# ── Step 4: map keys back to rows ───────────────────────────────────────
train_set = set(map(tuple, train_keys))
val_set   = set(map(tuple, val_keys))
test_set  = set(map(tuple, test_keys))

train_df = df2[df2['_pair_key'].isin(train_set)].copy()
val_df   = df2[df2['_pair_key'].isin(val_set)].copy()
test_df  = df2[df2['_pair_key'].isin(test_set)].copy()

# ── Step 5: drug-coverage fix ────────────────────────────────────────────
for _ in range(10):
    train_drugs = set(train_df['Drug 1'].unique()) | set(train_df['Drug 2'].unique())
    def bad_pairs(sdf):
        mask = ~(sdf['Drug 1'].isin(train_drugs) & sdf['Drug 2'].isin(train_drugs))
        return sdf.loc[mask, '_pair_key'].unique()
    bv, bt = bad_pairs(val_df), bad_pairs(test_df)
    if len(bv) == 0 and len(bt) == 0:
        break
    train_df = pd.concat([train_df,
                          val_df[val_df['_pair_key'].isin(bv)],
                          test_df[test_df['_pair_key'].isin(bt)]], ignore_index=True)
    val_df  = val_df[~val_df['_pair_key'].isin(bv)].copy()
    test_df = test_df[~test_df['_pair_key'].isin(bt)].copy()

for sdf in [train_df, val_df, test_df]:
    sdf.drop(columns=['_pair_key'], inplace=True)

# ── Verify & report ──────────────────────────────────────────────────────
total = len(train_df) + len(val_df) + len(test_df)
assert total == len(df), f'Row count mismatch: {total} vs {len(df)}'

print(f'{'Split':<8} {'Rows':>6}  {'%Total':>7}  {'%Severe':>8}')
print('-' * 35)
for name, sdf in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n = len(sdf)
    print(f'{name:<8} {n:>6}  {100*n/len(df):>6.1f}%  '
          f'{100*(sdf["Label"]==1).mean():>6.1f}%')

# ── Save splits ──────────────────────────────────────────────────────────
train_df.to_csv('split_train.csv', index=False)
val_df.to_csv('split_val.csv',     index=False)
test_df.to_csv('split_test.csv',   index=False)
print('\nSplit files saved.')

## 5. Encode the drugs as feature IDs

In [ ]:
# Build vocabulary from TRAINING split only.
# FM uses SEPARATE w1/w2 and v1/v2 tables → both indexed 0..n_drugs-1.
all_train_drugs = sorted(
    set(train_df['Drug 1'].unique()) | set(train_df['Drug 2'].unique()))
drug_to_id = {d: i for i, d in enumerate(all_train_drugs)}
n_drugs    = len(drug_to_id)

def encode(df):
    i1 = df['Drug 1'].map(drug_to_id).values.astype(np.int32)
    i2 = df['Drug 2'].map(drug_to_id).values.astype(np.int32)
    X  = np.stack([i1, i2], axis=1)   # shape (n, 2)
    y  = df['Label'].values.astype(np.float32)
    return X, y

X_train, y_train = encode(train_df)
X_val,   y_val   = encode(val_df)
X_test,  y_test  = encode(test_df)

print(f'n_drugs = {n_drugs}')
print(f'X_train shape: {X_train.shape}   y_train shape: {y_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'X_test shape:  {X_test.shape}')

## 6. The FM model

For Drug 1 = i and Drug 2 = j:

$$\hat{y} = b + w_1[i] + w_2[j] + \langle v_1[i],\, v_2[j] \rangle \qquad p = \sigma(\hat{y})$$

**Parameters:**
- `b` — scalar bias
- `w1`, `w2` — first-order weight vectors (shape `n_drugs`)
- `v1`, `v2` — embedding matrices (shape `n_drugs × k`)

In [ ]:
class FM:
    def __init__(self, n_drugs, k, seed=SEED):
        rng   = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(k)
        self.n_drugs = n_drugs
        self.k  = k
        self.b  = 0.0
        self.w1 = np.zeros(n_drugs, dtype=np.float64)
        self.w2 = np.zeros(n_drugs, dtype=np.float64)
        self.v1 = rng.uniform(-scale, scale, (n_drugs, k))
        self.v2 = rng.uniform(-scale, scale, (n_drugs, k))

    def forward(self, idx1, idx2):
        linear   = self.b + self.w1[idx1] + self.w2[idx2]
        dot_prod = np.sum(self.v1[idx1] * self.v2[idx2], axis=1)
        return linear + dot_prod

    def predict_proba(self, idx1, idx2):
        return sigmoid(self.forward(idx1, idx2))

    def loss(self, idx1, idx2, y):
        return bce_loss(self.predict_proba(idx1, idx2), y)

    def gradients(self, idx1, idx2, y):
        B = len(y)
        p = self.predict_proba(idx1, idx2)
        r = (p - y) / B     # residual, normalised

        grad_b  = float(r.sum())
        grad_w1 = np.zeros_like(self.w1)
        grad_w2 = np.zeros_like(self.w2)
        np.add.at(grad_w1, idx1, r)
        np.add.at(grad_w2, idx2, r)

        grad_v1 = np.zeros_like(self.v1)
        grad_v2 = np.zeros_like(self.v2)
        np.add.at(grad_v1, idx1, r[:, None] * self.v2[idx2])
        np.add.at(grad_v2, idx2, r[:, None] * self.v1[idx1])

        return {'b': grad_b, 'w1': grad_w1, 'w2': grad_w2,
                'v1': grad_v1, 'v2': grad_v2}

    def step(self, grads, lr):
        self.b  -= lr * grads['b']
        self.w1 -= lr * grads['w1']
        self.w2 -= lr * grads['w2']
        self.v1 -= lr * grads['v1']
        self.v2 -= lr * grads['v2']

    def save(self, path):
        np.savez(path, b=self.b, w1=self.w1, w2=self.w2,
                 v1=self.v1, v2=self.v2, n_drugs=self.n_drugs, k=self.k)

    @classmethod
    def load(cls, path):
        d = np.load(path)
        m = cls(int(d['n_drugs']), int(d['k']))
        m.b, m.w1, m.w2, m.v1, m.v2 = float(d['b']), d['w1'], d['w2'], d['v1'], d['v2']
        return m


def sigmoid(z):
    return np.where(z >= 0,
                    1.0 / (1.0 + np.exp(-z)),
                    np.exp(z) / (1.0 + np.exp(z)))

def bce_loss(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

print('FM class defined.')

## 7. Loss function — BCE demonstrated

In [ ]:
# Quick demo: BCE loss for one prediction
p_demo = np.array([0.7])   # model says 70% Severe
y_demo = np.array([1.0])   # actually Severe
print(f'p={p_demo[0]:.1f}, y={y_demo[0]:.0f}  =>  BCE = {bce_loss(p_demo, y_demo):.4f}')

p_demo2 = np.array([0.3])   # model says only 30% Severe
print(f'p={p_demo2[0]:.1f}, y={y_demo[0]:.0f}  =>  BCE = {bce_loss(p_demo2, y_demo):.4f}  (higher, worse)')

## 8. Gradients (the heart of "from scratch")

For mean BCE loss $L = -\frac{1}{B}\sum [y\log p + (1-y)\log(1-p)]$:

The sigmoid + BCE gradient cancels cleanly:
$$\frac{\partial L}{\partial \hat{y}} = \frac{p - y}{B}$$

Then by chain rule:
| Parameter | Gradient |
|---|---|
| $b$ | $\sum_n \frac{p_n - y_n}{B}$ |
| $w_1[i]$ | $\sum_{n:\,\text{drug}_1=i} \frac{p_n - y_n}{B}$ |
| $w_2[j]$ | $\sum_{n:\,\text{drug}_2=j} \frac{p_n - y_n}{B}$ |
| $v_1[i]$ | $\sum_{n:\,\text{drug}_1=i} \frac{p_n - y_n}{B} \cdot v_2[j_n]$ |
| $v_2[j]$ | $\sum_{n:\,\text{drug}_2=j} \frac{p_n - y_n}{B} \cdot v_1[i_n]$ |

In [ ]:
# ── Numerical gradient check ──────────────────────────────────────────────
def numerical_gradient_check(model, idx1, idx2, y, eps=1e-5, n_samples=6, seed=SEED):
    rng      = np.random.default_rng(seed)
    analytic = model.gradients(idx1, idx2, y)
    loss_fn  = lambda: model.loss(idx1, idx2, y)
    checks   = []

    def check_array(name, param, grad):
        fis = rng.choice(param.size, min(n_samples, param.size), replace=False)
        for fi in fis:
            orig = param.flat[fi]
            param.flat[fi] = orig + eps; lp = loss_fn()
            param.flat[fi] = orig - eps; lm = loss_fn()
            param.flat[fi] = orig
            num = (lp - lm) / (2 * eps)
            ana = grad.flat[fi]
            err = abs(num - ana) / max(abs(num), abs(ana), 1e-8)
            checks.append((name, fi, ana, num, err))

    # Bias handled separately (Python float)
    orig_b = model.b
    model.b = orig_b + eps; lp = loss_fn()
    model.b = orig_b - eps; lm = loss_fn()
    model.b = orig_b
    num_b = (lp - lm) / (2 * eps)
    err_b = abs(num_b - analytic['b']) / max(abs(num_b), abs(analytic['b']), 1e-8)
    checks.append(('b', 0, analytic['b'], num_b, err_b))

    check_array('w1', model.w1, analytic['w1'])
    check_array('w2', model.w2, analytic['w2'])
    check_array('v1', model.v1, analytic['v1'])
    check_array('v2', model.v2, analytic['v2'])

    max_err = max(c[4] for c in checks)
    print(f'Numerical gradient check  ({len(checks)} entries sampled)')
    print(f'  Max relative error: {max_err:.2e}')
    for name, fi, ana, num, err in checks:
        flag = '  <-- FAIL' if err > 1e-4 else ''
        print(f'  {name}[{fi}]  analytic={ana:+.6f}  numerical={num:+.6f}  '
              f'rel_err={err:.2e}{flag}')
    print('\n[PASS]' if max_err < 1e-4 else '\n[FAIL]')

# Run on a small real batch
rng_c  = np.random.default_rng(SEED)
idx_c  = rng_c.choice(len(y_train), 32, replace=False)
m_chk  = FM(n_drugs, k=4)
numerical_gradient_check(m_chk, X_train[idx_c, 0], X_train[idx_c, 1], y_train[idx_c])

## 9. Training loop

In [ ]:
def fm_f1(y_true, y_pred_prob, threshold=0.5):
    yhat = (y_pred_prob >= threshold).astype(np.float32)
    tp = int(((yhat==1)&(y_true==1)).sum())
    fp = int(((yhat==1)&(y_true==0)).sum())
    fn = int(((yhat==0)&(y_true==1)).sum())
    if tp == 0: return 0.0
    p = tp/(tp+fp); r = tp/(tp+fn)
    return 2*p*r/(p+r)


def train_fm(model, X_train, y_train, X_val, y_val,
             epochs=50, batch_size=256, lr=0.01, seed=SEED, verbose=True):
    rng  = np.random.default_rng(seed)
    N    = len(y_train)
    hist = {'train_loss': [], 'val_loss': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        perm   = rng.permutation(N)
        X_s, y_s = X_train[perm], y_train[perm]

        for start in range(0, N, batch_size):
            xb = X_s[start:start+batch_size]
            yb = y_s[start:start+batch_size]
            model.step(model.gradients(xb[:,0], xb[:,1], yb), lr)

        tl  = model.loss(X_train[:,0], X_train[:,1], y_train)
        p_v = model.predict_proba(X_val[:,0], X_val[:,1])
        vl  = bce_loss(p_v, y_val)
        vf  = fm_f1(y_val, p_v)

        hist['train_loss'].append(tl)
        hist['val_loss'].append(vl)
        hist['val_f1'].append(vf)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f'  Epoch {epoch:>3}/{epochs}  '
                  f'train={tl:.4f}  val={vl:.4f}  val_f1={vf:.4f}')
    return hist

print('Training functions defined.')

## 10. Evaluate on the validation set

In [ ]:
def best_threshold_f1(model, X_val, y_val):
    p = model.predict_proba(X_val[:,0], X_val[:,1])
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.65, 0.05):
        f1 = fm_f1(y_val, p, threshold=t)
        if f1 > best_f1:
            best_f1, best_t = f1, round(float(t), 2)
    return best_t, best_f1

def full_metrics(model, X, y, threshold):
    p    = model.predict_proba(X[:,0], X[:,1])
    yhat = (p >= threshold).astype(np.float32)
    loss = bce_loss(p, y)
    tp = int(((yhat==1)&(y==1)).sum())
    fp = int(((yhat==1)&(y==0)).sum())
    fn = int(((yhat==0)&(y==1)).sum())
    tn = int(((yhat==0)&(y==0)).sum())
    acc  = (tp+tn)/len(y)
    prec = tp/(tp+fp) if tp+fp > 0 else 0.0
    rec  = tp/(tp+fn) if tp+fn > 0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec > 0 else 0.0
    return {'loss':loss,'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,
            'tp':tp,'fp':fp,'fn':fn,'tn':tn}

print('Evaluation helpers defined.')

## 11. Experiment: embedding dimension k

In [ ]:
K_VALUES   = [4, 8, 16, 32]
EPOCHS     = 50
BATCH_SIZE = 256
LR         = 0.01

all_histories = {}
all_results   = []
best_models   = {}

for k in K_VALUES:
    print(f'\n{"="*50}')
    print(f'Training FM  k={k}')
    print('='*50)
    model = FM(n_drugs, k=k, seed=SEED)
    hist  = train_fm(model, X_train, y_train, X_val, y_val,
                     epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR)

    threshold, _ = best_threshold_f1(model, X_val, y_val)
    m = full_metrics(model, X_val, y_val, threshold)
    n_params = (1 + model.w1.size + model.w2.size
                + model.v1.size + model.v2.size)

    row = {'k': k, 'threshold': threshold,
           'val_loss': round(m['loss'],6),
           'val_accuracy': round(m['accuracy'],6),
           'val_precision': round(m['precision'],6),
           'val_recall': round(m['recall'],6),
           'val_f1': round(m['f1'],6),
           'n_params': n_params}
    all_results.append(row)
    all_histories[k] = hist
    best_models[k]   = model

    print(f'  → threshold={threshold}  val_f1={m["f1"]:.4f}  '
          f'val_acc={m["accuracy"]:.4f}  prec={m["precision"]:.4f}  '
          f'rec={m["recall"]:.4f}  params={n_params}')
    model.save(f'fm_k{k}.npz')

# ── Summary table ─────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
print('\n', results_df.to_string(index=False))

results_df.to_csv('fm_experiments.csv', index=False)
print('\nResults saved to fm_experiments.csv')

best_k = results_df.loc[results_df['val_f1'].idxmax(), 'k']
print(f'\nBest FM: k={best_k}  '
      f'(val F1 = {results_df[results_df["k"]==best_k]["val_f1"].values[0]:.4f})')

## Plot: training vs validation loss

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, k in zip(axes, K_VALUES):
    hist = all_histories[k]
    ep   = range(1, len(hist['train_loss'])+1)
    ax.plot(ep, hist['train_loss'], label='Train loss', color='#1f77b4')
    ax.plot(ep, hist['val_loss'],   label='Val loss',   color='#ff7f0e', ls='--')
    ax.set_title(f'FM  k={k}', fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('FM — Train vs Validation Loss  (lr=0.01, batch=256)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fm_loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved: fm_loss_curves.png')

## Plot: validation F1 vs k

In [ ]:
ks  = [r['k']      for r in all_results]
f1s = [r['val_f1'] for r in all_results]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([str(k) for k in ks], f1s, color='#2196F3',
              edgecolor='black', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=10)
ax.set_xlabel('Embedding size  k', fontsize=12)
ax.set_ylabel('Validation F1  (Severe class)', fontsize=12)
ax.set_title(f'FM — Validation F1 vs k   (best: k={best_k})', fontsize=13)
ax.set_ylim(0, max(f1s)*1.3 + 0.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fm_val_f1_vs_k.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved: fm_val_f1_vs_k.png')

## 12. Final test evaluation (run once)

In [ ]:
# Load the chosen model (best_k selected above from validation F1)
final_model = best_models[best_k]
final_threshold = results_df[results_df['k']==best_k]['threshold'].values[0]

m_test = full_metrics(final_model, X_test, y_test, final_threshold)

print('=' * 50)
print(f'TEST SET EVALUATION  (k={best_k}, threshold={final_threshold})')
print('=' * 50)
print(f'  Accuracy  : {m_test["accuracy"]:.4f}')
print(f'  Precision : {m_test["precision"]:.4f}')
print(f'  Recall    : {m_test["recall"]:.4f}')
print(f'  F1        : {m_test["f1"]:.4f}')
print(f'  Loss      : {m_test["loss"]:.4f}')
print()
print('Confusion Matrix:')
print(f'  TP={m_test["tp"]}  FP={m_test["fp"]}')
print(f'  FN={m_test["fn"]}  TN={m_test["tn"]}')

## 13. Reasoning task: explain one prediction

In [ ]:
# Pick a drug pair the model predicts as Severe (ideally a true positive)
p_test = final_model.predict_proba(X_test[:,0], X_test[:,1])
tp_idx = np.where((p_test >= final_threshold) & (y_test == 1))[0]

if len(tp_idx) == 0:
    print('No true positives found at this threshold. Using any predicted Severe.')
    tp_idx = np.where(p_test >= final_threshold)[0]

chosen = tp_idx[0]
i = int(X_test[chosen, 0])
j = int(X_test[chosen, 1])

# Look up drug names
id_to_drug = {v: k for k, v in drug_to_id.items()}
drug1_name = id_to_drug[i]
drug2_name = id_to_drug[j]

bias_term  = final_model.b
w1_term    = final_model.w1[i]
w2_term    = final_model.w2[j]
dot_term   = float(np.dot(final_model.v1[i], final_model.v2[j]))
logit      = bias_term + w1_term + w2_term + dot_term
prob       = float(sigmoid(np.array([logit]))[0])
true_label = int(y_test[chosen])

print(f'Drug 1 : {drug1_name}  (ID {i})')
print(f'Drug 2 : {drug2_name}  (ID {j})')
print(f'True label: {true_label}  (1=Severe)')
print()
print(f'Score breakdown:')
print(f'  bias       w0          = {bias_term:+.6f}')
print(f'  linear     w1[{i}]     = {w1_term:+.6f}')
print(f'  linear     w2[{j}]     = {w2_term:+.6f}')
print(f'  interaction dot(v1,v2) = {dot_term:+.6f}')
print(f'  ─────────────────────────────────')
print(f'  logit  z               = {logit:+.6f}')
print(f'  probability p=σ(z)     = {prob:.6f}')
print(f'  prediction (≥{final_threshold}) = {int(prob >= final_threshold)}')

## 14. Results (standard format, required)

In [ ]:
fm_results = {
    'model':           'FM (from scratch)',
    'split_ratio':     '70/15/15',
    'embedding_dim_k': int(best_k),
    'learning_rate':   LR,
    'epochs':          EPOCHS,
    'threshold':       float(final_threshold),
    'val_f1':          round(results_df[results_df['k']==best_k]['val_f1'].values[0], 6),
    'test_accuracy':   round(m_test['accuracy'],  6),
    'test_precision':  round(m_test['precision'], 6),
    'test_recall':     round(m_test['recall'],    6),
    'test_f1':         round(m_test['f1'],        6),
}
print(fm_results)

pd.DataFrame([fm_results]).to_csv('fm_results.csv', index=False)
print('\nSaved: fm_results.csv  (load this in Notebook 2)')